# 策略研发全流程 Pipeline

本 Notebook 是量化策略研发的统一核心入口，严格遵循 `strategy_rules.md` 中的样本内外划分与无前视偏差规则：
- **IS_Train (滚动训练集)**: `2021-01-01` ~ `2023-09-30`，仅用于 Rolling WFV 内部早停与调参。
- **IS_Test (核心测试集)**: `2023-10-01` ~ `2024-09-01`，使用最近 4-Fold 集成进行验证，决定策略是否可以进入实盘预备。
- **OOS (严格样本外)**: `2024-09-01` 之后，作为盲区进行最后一道防线检验。

In [1]:
import sys
import logging
from pathlib import Path

# 配置项目根目录
ROOT = Path("/root/autodl-tmp/Strategy")
sys.path.insert(0, str(ROOT.parent))

logging.basicConfig(level=logging.INFO, force=True)
for name in ("Strategy", "Strategy.model", "Strategy.backtest"):
    logging.getLogger(name).setLevel(logging.INFO)

from Strategy import config


## 1. 数据准备 (Label & Factors)
在全市场范围内生成所需的 Label 和因子。本 Notebook **训练默认标签 `OPEN930_1000`**：买入价为连续竞价 **首根 K 线**（本仓库分钟数据 `time=931`，与 `OPEN0935` 同理均为真实 bar）→ T+1 **10:00** 卖出（见 `label_generator`）。依赖 **`min_data`** 与 **`Daily_data`**。因子对齐 **T-1 收盘后** 可得。可改用 **`OPEN0935_1000`** 或 **`CLOSE_PRECLOSE`**。


In [ ]:
from Strategy.label.label_generator import generate_and_save_open930_1000_label

LABEL_TAG = "OPEN930_1000"
# 需 min_data 分钟线；Daily_data: CLOSE/PRE_CLOSE（除权）+ LIMIT_UP + LIMIT_DOWN（默认涨跌停剔除）
# 可选: start_date=20210104, end_date=20251231；调试可无涨跌停: use_limit_price_tables=False
open930_label_path = generate_and_save_open930_1000_label()
print(f"[{LABEL_TAG}] Label 已生成: {open930_label_path}")

# 其他示例:
# from Strategy.label.label_generator import generate_and_save_open0935_1000_label, generate_and_save_close_preclose_label
# generate_and_save_open0935_1000_label(save_price_tables=True)
# generate_and_save_close_preclose_label()


In [ ]:
# from Strategy.factor.daily_factor import DailyFactorLibraryAdapter

# # 日频因子需覆盖到当前可用的最新日期
# daily_adapter = DailyFactorLibraryAdapter()
# saved_daily = daily_adapter.compute_and_save_all(
#     start_date=config.IS_TRAIN_START,
#     # end_date=config.IS_TRAIN_END,  # 仅测试时取消注释以加速
# )
# print(f"已保存 {len(saved_daily)} 个日频因子")

In [ ]:
from Strategy.factor.min_factor import MinFactorAdapter
# 分钟因子使用 min_data 原始分钟 K 线，取 931-1457 并仅保留 1457 截面
min_adapter = MinFactorAdapter()
saved_min = min_adapter.compute_and_save_all(
    start_date=config.IS_TRAIN_START,
    # end_date=config.IS_TRAIN_END,  # 仅测试时取消注释以加速
    n_workers=8,                  # 可按机器核数调整
)
print(f"已保存 {len(saved_min)} 个分钟因子")

INFO:Strategy.factor.min_factor:MinFactor: processing 1261 daily minute files (multiprocess=True, workers=8)


MinFactor 按日计算:   0%|          | 0/1261 [00:00<?, ?day/s]

In [ ]:
# from Strategy.factor.factor_base import FactorRegistry
# import Strategy.factor.daily_factor             # 触发纯日频注册因子

# # 计算并保存所有注册的纯日频扩展因子
# FactorRegistry.compute_all()

# # 分钟字段主流程已统一改为 Strategy.factor.min_factor.MinFactorAdapter


## 1.1 GP 因子挖掘
在基础因子 `.fea` 宽表生成后，可选运行 Strategy-native GP 框架，为当前 `LABEL_TAG` 挖掘派生因子。GP 只使用 Train+IS Test 做筛选，OOS 仅保留为报告口径，不参与选择。


In [ ]:
from pathlib import Path

from Strategy.gp_mining.run_gp import mine_gp_factors

# GP 搜索会调用 torch 算子并逐候选做截面评估，正式运行前建议先用小规模参数烟测。
RUN_GP_MINING = False
INCLUDE_GP_FACTORS_IN_PANEL = RUN_GP_MINING  # 若要复用已存在的 GP 因子目录，可手动改为 True
GP_EXPERIMENT = f"gp_{LABEL_TAG}_v1"
GP_TERMINAL_NAMES = None  # None 表示使用 outputs/factors/*.fea 全部基础因子；调试可填 ["ma5", "ma10", "rsi_6"]

if RUN_GP_MINING:
    gp_result = mine_gp_factors(
        label_tag=LABEL_TAG,
        experiment=GP_EXPERIMENT,
        generations=32,
        population=64,
        terminal_names=GP_TERMINAL_NAMES,
        save_factors=True,
        overwrite=True,
        device="auto",
        parent_selection="epsilon_lexicase",
        epsilon_lexicase_epsilon=0.002,
        lexicase_case_sample_ratio=0.35,
    )
    print(f"GP 挖掘完成: evaluated={len(gp_result.leaderboard_rows)}, accepted={len(gp_result.accepted_rows)}")
else:
    print("GP 因子挖掘已跳过。将 RUN_GP_MINING 改为 True 后运行本 cell。")

GP_FACTOR_DIR = Path("/root/autodl-tmp/Strategy/outputs/gp_factors") / GP_EXPERIMENT
print(f"GP 因子目录: {GP_FACTOR_DIR}")


## 2. 面板构建与样本拆分
严格按照配置，物理隔离出 `is_train`, `is_test`, `oos`。模型训练时将绝不碰触 `is_test` 和 `oos`。

In [ ]:
from pathlib import Path

import pandas as pd

from Strategy.factor.factor_base import load_all_factors
from Strategy.label.label_generator import load_label
from Strategy.model.trainer import build_panel, split_panel
from Strategy.utils.helpers import ensure_tradedate_as_index

LABEL_TAG = globals().get("LABEL_TAG", "OPEN930_1000")
factor_dict = load_all_factors()

# 若上一步运行过 GP 因子挖掘，且 INCLUDE_GP_FACTORS_IN_PANEL=True，则将 GP 因子并入训练特征。
gp_factor_dir = globals().get("GP_FACTOR_DIR")
if globals().get("INCLUDE_GP_FACTORS_IN_PANEL", False) and gp_factor_dir is not None:
    gp_factor_dir = Path(gp_factor_dir)
    if gp_factor_dir.exists():
        gp_factor_paths = sorted(gp_factor_dir.glob("*.fea"))
        for path in gp_factor_paths:
            factor_dict[f"gp__{path.stem}"] = ensure_tradedate_as_index(pd.read_feather(path))
        print(f"已加载 {len(gp_factor_paths)} 个 GP 因子: {gp_factor_dir}")

label_df = load_label(LABEL_TAG)

# 展平构建 Panel
panel = build_panel(factor_dict, label_df)

# 严格切割样本 (依据 config 中定好的常数)
is_train, is_test, oos = split_panel(panel)

print(f"IS_Train: {is_train.shape[0]} rows (截止 {is_train['TRADE_DATE'].max().date()})")
print(f"IS_Test:  {is_test.shape[0]} rows (截止 {is_test['TRADE_DATE'].max().date()})")
print(f"OOS:      {oos.shape[0]} rows (起始 {oos['TRADE_DATE'].min().date()})")


## 3. IS Train 滚动训练 (Rolling CV)
利用 `is_train` 内部通过时间窗滚动的方式进行训练和早停，并保存模型参数。
与 `strategy_rules.md` 一致：**截面模型**、**`RollingTrainer` 时间块 CV**、神经网络 **`batch_size=1`**（每个优化步仅一个交易日截面；见 `config.NN_TRAINER_BATCH_SIZE`）。


In [ ]:
from Strategy.model.rolling_trainer import RollingTrainer
from Strategy.model.trainer import XGBTrainer

rt_xgb = RollingTrainer(
    model_class=XGBTrainer,
    model_kwargs={
        "use_wpcc": True,
    }
)

# 仅在 IS_Train 上执行所有 Fold 的滚动训练
rt_xgb.train_all_folds(is_train)

print("\n========== XGB CV Fold IC ==========")
print(rt_xgb.fold_ic_report())

rt_xgb.save_model(config.SCORE_OUTPUT_DIR / f"rolling_xgb_{LABEL_TAG}.pkl")


In [ ]:
from Strategy.model.transformer_trainer import TransformerTrainer

rt_tf = RollingTrainer(
    model_class=TransformerTrainer,
    model_kwargs={
        "d_model": 64, "nhead": 4, "num_layers": 2, "d_ff": 128, "dropout": 0.15,
        "epochs": 50, "lr": 5e-4, "weight_decay": 0.01, "batch_size": 1,
        "early_stopping_patience": 8, "device": "cuda", "use_amp": False,
    }
)

rt_tf.train_all_folds(is_train)

print("\n========== Transformer CV Fold IC ==========")
print(rt_tf.fold_ic_report())
rt_tf.save_model(config.SCORE_OUTPUT_DIR / f"rolling_transformer_{LABEL_TAG}.pkl")


In [ ]:
from Strategy.model.mlp_trainer import MLPTrainer

rt_mlp = RollingTrainer(
    model_class=MLPTrainer,
    model_kwargs={
        "hidden_dims": (256, 128, 64),
        "dropout": 0.15,
        "epochs": 50,
        "lr": 5e-4,
        "weight_decay": 0.01,
        "batch_size": 1,
        "early_stopping_patience": 8,
        "device": "cuda",
        "use_amp": False,
    },
)

rt_mlp.train_all_folds(is_train)

print("\n========== MLP CV Fold IC ==========")
print(rt_mlp.fold_ic_report())
rt_mlp.save_model(config.SCORE_OUTPUT_DIR / f"rolling_mlp_{LABEL_TAG}.pkl")


## 4. IS Test 核心推理与模型集成
利用最近的 4-Fold 模型对 `is_test` 进行打分，随后进行跨模型种类（XGB / Transformer / MLP）的集成。


In [ ]:
from Strategy.model.scorer import mask_scores_by_price
from Strategy.data_io.saver import save_wide_table
from Strategy.model.ensemble_scorer import compute_score_correlation, select_ensemble_models, ensemble_scores

score_test_xgb = rt_xgb.predict_is_test(is_test, normalize=True)
score_test_xgb = mask_scores_by_price(score_test_xgb, label_tag=LABEL_TAG)
save_wide_table(score_test_xgb, config.SCORE_OUTPUT_DIR / f"SCORE_xgb_{LABEL_TAG}_is_test.fea")

score_test_tf = rt_tf.predict_is_test(is_test, normalize=True)
score_test_tf = mask_scores_by_price(score_test_tf, label_tag=LABEL_TAG)
save_wide_table(score_test_tf, config.SCORE_OUTPUT_DIR / f"SCORE_transformer_{LABEL_TAG}_is_test.fea")

score_test_mlp = rt_mlp.predict_is_test(is_test, normalize=True)
score_test_mlp = mask_scores_by_price(score_test_mlp, label_tag=LABEL_TAG)
save_wide_table(score_test_mlp, config.SCORE_OUTPUT_DIR / f"SCORE_mlp_{LABEL_TAG}_is_test.fea")

score_dfs = {
    "xgb": score_test_xgb,
    "transformer": score_test_tf,
    "mlp": score_test_mlp,
}

corr = compute_score_correlation(score_dfs)
print("\n模型打分截面相关性矩阵:")
print(corr)

selected = select_ensemble_models(score_dfs, max_pairwise_corr=0.9)

score_test_ens = ensemble_scores(
    score_dfs,
    selected_models=selected,
    label_tag=f"{LABEL_TAG}_is_test",
    save=True,
    output_dir=config.SCORE_OUTPUT_DIR
)


## 5. 快速业绩回测 (仅观测 IS_Test)
在没有发生前视偏差的数据区间上观测策略真实的盈利能力。

In [ ]:
import sys
import logging
from pathlib import Path

ROOT = Path("/root/autodl-tmp/Strategy")
if str(ROOT.parent) not in sys.path:
    sys.path.insert(0, str(ROOT.parent))
logging.basicConfig(level=logging.INFO, force=True)

from Strategy import config
from Strategy.label.label_generator import load_label
from Strategy.model.scorer import load_is_test_scores_from_disk
from Strategy.backtest.quick_backtest import run_quick_backtest

LABEL_TAG = "OPEN930_1000"

label_df = load_label(LABEL_TAG)
models_to_test = load_is_test_scores_from_disk(LABEL_TAG)

for name, score_df in models_to_test.items():
    print(f"\n{'='*50}\n回测评估: {name} (IS_Test)\n{'='*50}")
    run_quick_backtest(
        score_df=score_df,
        label_df=label_df,
        n_groups=20,
        output_dir=config.BT_RESULT_DIR / "is_test" / name,
        start_date=config.IS_TEST_START,
        top_ks=(20, 50, 100),
        tail_ks=(20, 50, 100),
        run_ic=True,
        title=f"{name.upper()} | {LABEL_TAG} | IS_Test",
    )


## 6. OOS 盲区验证 (严格门控)
⚠️ **重要提示**：此部分代码仅当上方 `IS_Test` 回测结果各项指标完全达到预期目标，决定实盘部署时，才允许运行！一旦针对 OOS 数据调整超参，OOS 就被“污染”了。

In [ ]:
# score_oos_xgb = rt_xgb.predict_is_test(oos, normalize=True)
# score_oos_tf = rt_tf.predict_is_test(oos, normalize=True)
# score_oos_mlp = rt_mlp.predict_is_test(oos, normalize=True)
# score_oos_xgb = mask_scores_by_price(score_oos_xgb, label_tag=LABEL_TAG)
# score_oos_tf = mask_scores_by_price(score_oos_tf, label_tag=LABEL_TAG)
# score_oos_mlp = mask_scores_by_price(score_oos_mlp, label_tag=LABEL_TAG)
#
# score_oos_ens = ensemble_scores(
#     {"xgb": score_oos_xgb, "transformer": score_oos_tf, "mlp": score_oos_mlp},
#     selected_models=selected,
#     label_tag=f"{LABEL_TAG}_oos",
#     save=True,
#     output_dir=config.SCORE_OUTPUT_DIR
# )
#
# run_quick_backtest(
#     score_df=score_oos_ens,
#     label_df=label_df,
#     n_groups=20,
#     output_dir=config.BT_RESULT_DIR / "oos" / "ensemble",
#     start_date=config.OOS_START,
#     top_ks=(20, 50, 100),
#     tail_ks=(20, 50, 100),
#     run_ic=True,
#     title=f"ENSEMBLE | {LABEL_TAG} | OOS"
# )
